In [ ]:
import pandas as pd
import numpy as np

financial_df = pd.read_csv("./csv/customer_financial_health_with_ai.csv")
customers_df = pd.read_csv("./csv/customers.csv")
purchases_df = pd.read_csv("./csv/home_purchases.csv")
loans_df = pd.read_csv("./csv/loans_raw.csv")
estate_detail_df = pd.read_csv("./csv/real_estate_detail_raw.csv")
estate_df = pd.read_csv("./csv/real_estate_raw.csv")

pd.options.display.float_format = '{:,.0f}'.format

In [ ]:
single_df = purchases_df[
    purchases_df["household_size"] == 1
].copy()

In [ ]:
# primary vs household
print(
    "두 소득 동일:",
    (single_df["primary_annual_income"] ==
     single_df["household_annual_income"]).sum()
)

print(
    "두 소득 다름:",
    (single_df["primary_annual_income"] !=
     single_df["household_annual_income"]).sum()
)

In [ ]:
single_df["household_primary_ratio"] = (
    single_df["household_annual_income"] /
    single_df["primary_annual_income"]
)

0.5	가구소득이 개인소득의 0.5배  
1.0	두 소득 동일  
1.5	가구소득이 개인소득의 1.5배  
2.0	가구소득이 개인소득의 2배  
3.0	가구소득이 개인소득의 3배  
4.0	가구소득이 개인소득의 4배  
5.0	가구소득이 개인소득의 5배  

In [ ]:
bins = [0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, np.inf]

labels = [
    "0~0.5배",
    "0.5~1배",
    "1~1.5배",
    "1.5~2배",
    "2~3배",
    "3~4배",
    "4~5배",
    "5배 이상"
]

single_df["income_ratio_group"] = pd.cut(
    single_df["household_primary_ratio"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

ratio_result = (
    single_df["income_ratio_group"]
    .value_counts()
    .sort_index()
    .to_frame("건수")
)

ratio_result["비율(%)"] = (
    ratio_result["건수"] / ratio_result["건수"].sum() * 100
).round(2)

ratio_result

In [ ]:
outlier_4x = single_df[
    single_df["household_primary_ratio"] >= 4
]

print("1인 가구 전체:", len(single_df))
print("4배 이상:", len(outlier_4x))
print(
    "4배 이상 비율:",
    round(len(outlier_4x) / len(single_df) * 100, 2),
    "%"
)

outlier_4x[
    [
        "customer_id",
        "primary_annual_income",
        "household_annual_income",
        "household_primary_ratio"
    ]
].sort_values(
    "household_primary_ratio",
    ascending=False
).head(10)

In [ ]:
high_ratio = single_df[
    single_df["household_primary_ratio"] >= 4
].copy()

high_ratio[
    [
        "customer_id",
        "primary_annual_income",
        "household_annual_income",
        "household_primary_ratio",
        "employment_type",
        "age"
    ]
].sort_values(
    "household_primary_ratio",
    ascending=False
).head(5)

In [ ]:
high_ratio[
    [
        "primary_annual_income",
        "household_annual_income",
        "household_primary_ratio"
    ]
].describe()

In [ ]:
#더블체크
single_df.groupby(
    pd.cut(
        single_df["household_primary_ratio"],
        bins=[0, 1, 2, 3, 4, 5, np.inf],
        labels=["1배 이하", "1~2배", "2~3배", "3~4배", "4~5배", "5배 이상"]
    ),
    observed=False
)[
    ["primary_annual_income", "household_annual_income"]
].median()

1인 가구에서도 개인소득 대비 가구소득이 크게 상회하는 사례가 다수 확인되었으며, 소득비율 증가에 따라 개인소득은 감소하고 가구소득은 증가하는 패턴이 관찰되었다. 따라서 임의의 배수 기준으로 이상치를 제거하기보다, 가구소득 변수의 산출 기준을 우선 확인한 후 모델 적용 여부를 결정할 필요가 있다.

전체의 10프로로 삭제해도 데이터양이 많아서 제거고려.  
    -> 제거 전/후 모델을 둘 다 돌려서 성능을 비교하는 방식

In [ ]:
# ratio 추가
model_df = purchases_df.copy()

model_df["household_primary_ratio"] = (
    model_df["household_annual_income"] /
    model_df["primary_annual_income"]
)

In [ ]:
model_df.to_csv(
    "./csv/home_purchases_model_base.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# 1인가구 소득 4배이상 삭제버전
remove_condition = (
    (model_df["household_size"] == 1) &
    (model_df["household_primary_ratio"] >= 4)
)

model_df_filtered = model_df[
    ~remove_condition
].copy()

print("제거 전:", len(model_df))
print("제거 건수:", remove_condition.sum())
print("제거 후:", len(model_df_filtered))
print(
    "전체 대비 제거 비율:",
    round(remove_condition.mean() * 100, 2), "%"
)

In [ ]:
model_df_filtered.to_csv(
    "./csv/home_purchases_model_filtered.csv",
    index=False,
    encoding="utf-8-sig"
)